# 消费信贷违约风险分析｜SQL / SQLite

## 项目说明
本 Notebook 使用与 Python 分析相同的消费信贷数据，通过 SQL 提取并验证核心风险指标。

SQL 部分不重复全部 Python 分析，而是重点展示：
- 基础风险指标统计；
- `CASE WHEN` 风险分组；
- `GROUP BY` 分组违约率分析；
- 多月逾期次数构造；
- 最大逾期程度构造；
- `AND / OR` 组合条件与连续逾期识别。

## 分析框架
`数据入库 → 基础指标 → 单月逾期 → 近6个月逾期频率 → 最大逾期程度 → 连续逾期`

> 数据库环境：SQLite  
> SQL 查询通过 `pd.read_sql_query()` 在 Jupyter Notebook 中执行。

## 1. 数据准备与 SQLite 环境

为了使本 Notebook 可以独立运行，这里重复最小必要的数据读取和字段重命名步骤，然后将 DataFrame 写入 SQLite 内存数据库。

数据库表名统一为：

`credit_risk`

In [1]:
import pandas as pd
import sqlite3
from ucimlrepo import fetch_ucirepo

In [2]:
credit_dataset = fetch_ucirepo(id=350)

X = credit_dataset.data.features.copy()
y = credit_dataset.data.targets.copy()

raw_df = pd.concat([X, y], axis=1)

In [3]:
target_column = y.columns[0]

raw_df.rename(
    columns={target_column: "default_flag"},
    inplace=True
)

column_mapping = {
    "X1": "credit_limit",
    "X2": "sex",
    "X3": "education",
    "X4": "marriage",
    "X5": "age",
    "X6": "repay_status_sep",
    "X7": "repay_status_aug",
    "X8": "repay_status_jul",
    "X9": "repay_status_jun",
    "X10": "repay_status_may",
    "X11": "repay_status_apr",
    "X12": "bill_amount_sep",
    "X13": "bill_amount_aug",
    "X14": "bill_amount_jul",
    "X15": "bill_amount_jun",
    "X16": "bill_amount_may",
    "X17": "bill_amount_apr",
    "X18": "payment_amount_sep",
    "X19": "payment_amount_aug",
    "X20": "payment_amount_jul",
    "X21": "payment_amount_jun",
    "X22": "payment_amount_may",
    "X23": "payment_amount_apr"
}

raw_df.rename(columns=column_mapping, inplace=True)

credit_df = raw_df.copy()

In [4]:
conn = sqlite3.connect(":memory:")

credit_df.to_sql(
    "credit_risk",
    conn,
    if_exists="replace",
    index=False
)

30000

### 1.1 检查 SQL 表

先查看前5条记录，确认 DataFrame 已成功写入 `credit_risk` 表。

In [5]:
pd.read_sql_query(
    """
    SELECT *
    FROM credit_risk
    LIMIT 5;
    """,
    conn
)

,credit_limit,sex,education,marriage,age,repay_status_sep,repay_status_aug,repay_status_jul,repay_status_jun,repay_status_may,...,bill_amount_jun,bill_amount_may,bill_amount_apr,payment_amount_sep,payment_amount_aug,payment_amount_jul,payment_amount_jun,payment_amount_may,payment_amount_apr,default_flag
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 2. 整体违约风险指标

**分析问题：**
数据库中共有多少客户、多少违约客户，整体下一期违约率是多少？

由于 `default_flag` 为0/1变量：
- `COUNT(*)`：客户总数；
- `SUM(default_flag)`：违约客户数；
- `AVG(default_flag)`：整体违约率。

In [6]:
pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS customer_count,
        SUM(default_flag) AS default_customer_count,
        AVG(default_flag) AS default_rate
    FROM credit_risk;
    """,
    conn
)

,customer_count,default_customer_count,default_rate
0,30000,6636,0.2212


**分析结果：**
- 客户总数：30,000
- 下一期违约客户数：6,636
- 整体下一期违约率：22.12%

这一查询展示了 SQL 中 `COUNT`、`SUM` 和 `AVG` 在0/1目标变量上的基础风险统计应用。

## 3. 9月逾期状态与下一期违约风险

### 3.1 是否逾期

**分析问题：**
最近一期已经出现逾期的客户，下一期违约率是否明显更高？

使用 `CASE WHEN` 将客户分为：
- `repay_status_sep > 0`：9月逾期；
- 其他状态：9月未逾期。

再通过 `GROUP BY` 分组计算客户数和下一期违约率。

In [7]:
pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN repay_status_sep > 0 THEN '9月逾期'
            ELSE '9月未逾期'
        END AS overdue_status,

        COUNT(*) AS customer_count,
        AVG(default_flag) AS default_rate

    FROM credit_risk
    GROUP BY overdue_status;
    """,
    conn
)

,overdue_status,customer_count,default_rate
0,9月未逾期,23182,0.138340
1,9月逾期,6818,0.502933


**分析结果：**
- 9月未逾期客户：23,182人，违约率约13.83%
- 9月逾期客户：6,818人，违约率约50.29%

9月逾期客户的下一期违约率显著高于未逾期客户，说明最近一期逾期状态具有较强的风险区分能力。

### 3.2 逾期严重程度

进一步将9月还款状态划分为：
- 未逾期；
- 逾期1个月；
- 逾期2个月；
- 逾期3个月及以上。

观察逾期程度加重后违约率是否上升。

In [8]:
pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN repay_status_sep <= 0 THEN '未逾期'
            WHEN repay_status_sep = 1 THEN '逾期1个月'
            WHEN repay_status_sep = 2 THEN '逾期2个月'
            ELSE '逾期3个月及以上'
        END AS overdue_level,

        COUNT(*) AS customer_count,
        AVG(default_flag) AS default_rate

    FROM credit_risk
    GROUP BY overdue_level;
    """,
    conn
)

,overdue_level,customer_count,default_rate
0,未逾期,23182,0.138340
1,逾期1个月,3688,0.339479
2,逾期2个月,2667,0.691414
3,逾期3个月及以上,463,0.719222


**分析结果：**
- 未逾期：13.83%
- 逾期1个月：33.95%
- 逾期2个月：69.14%
- 逾期3个月及以上：71.92%

整体上，逾期严重程度越高，下一期违约率越高。

## 4. 近6个月逾期频率

**分析问题：**
近6个月发生逾期的月份越多，下一期违约风险是否越高？

对6个月还款状态分别使用：

`CASE WHEN repay_status > 0 THEN 1 ELSE 0 END`

再将6个0/1结果相加，得到每个客户近6个月的逾期月份数。

In [9]:
pd.read_sql_query(
    """
    SELECT
        (CASE WHEN repay_status_sep > 0 THEN 1 ELSE 0 END)
        +
        (CASE WHEN repay_status_aug > 0 THEN 1 ELSE 0 END)
        +
        (CASE WHEN repay_status_jul > 0 THEN 1 ELSE 0 END)
        +
        (CASE WHEN repay_status_jun > 0 THEN 1 ELSE 0 END)
        +
        (CASE WHEN repay_status_may > 0 THEN 1 ELSE 0 END)
        +
        (CASE WHEN repay_status_apr > 0 THEN 1 ELSE 0 END)
        AS overdue_month_count_6m,

        COUNT(*) AS customer_count,
        AVG(default_flag) AS default_rate

    FROM credit_risk
    GROUP BY overdue_month_count_6m;
    """,
    conn
)

,overdue_month_count_6m,customer_count,default_rate
0,0,19931,0.117104
1,1,4426,0.298238
2,2,1899,0.387572
3,3,1154,0.508666
4,4,951,0.573081
5,5,298,0.573826
6,6,1341,0.703207


**分析结果：**

近6个月逾期月份数从0增加到6时，下一期违约率整体由约11.71%上升至70.32%。

这说明历史逾期频率与后续违约风险具有明显关联，也是较有解释力的风险特征。

## 5. 近6个月最大逾期程度

**分析问题：**
客户近6个月最严重的一次逾期状态，能否反映下一期违约风险？

在 SQLite 中，`MAX(a, b, c, ...)` 可以比较同一行中的多个值。因此这里将6个月还款状态横向比较，并额外加入 `0`，使负数状态最低按0处理。

> 注意：这里使用的是 SQLite 支持的多参数 `MAX()` 标量函数写法，不是所有数据库都使用相同语法。

In [10]:
pd.read_sql_query(
    """
    SELECT
        MAX(
            0,
            repay_status_sep,
            repay_status_aug,
            repay_status_jul,
            repay_status_jun,
            repay_status_may,
            repay_status_apr
        ) AS max_overdue_months_6m,

        COUNT(*) AS customer_count,
        AVG(default_flag) AS default_rate

    FROM credit_risk
    GROUP BY max_overdue_months_6m;
    """,
    conn
)

,max_overdue_months_6m,customer_count,default_rate
0,0,19931,0.117104
1,1,1689,0.249852
2,2,7187,0.435509
3,3,789,0.622307
4,4,218,0.642202
5,5,69,0.507246
6,6,25,0.560000
7,7,67,0.835821
8,8,25,0.560000


**分析结果：**

最大逾期程度从0提高至4个月时，下一期违约率由约11.71%上升至64.22%。

5个月以上组别样本量明显较小，违约率存在波动，因此高等级结果需要结合客户数量谨慎解释。

## 6. 连续逾期与下一期违约风险

**分析问题：**
若客户在任意两个相邻月份均发生逾期，是否表现出更高的后续违约风险？

连续逾期的判断逻辑为：

- 相邻两个月同时逾期：使用 `AND`
- 5组相邻月份中任意一组成立：使用 `OR`

最终构造 `continuous_overdue_flag_6m`：
- `1`：近6个月存在连续逾期；
- `0`：近6个月不存在连续逾期。

In [11]:
pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN
                (repay_status_sep > 0 AND repay_status_aug > 0)
                OR (repay_status_aug > 0 AND repay_status_jul > 0)
                OR (repay_status_jul > 0 AND repay_status_jun > 0)
                OR (repay_status_jun > 0 AND repay_status_may > 0)
                OR (repay_status_may > 0 AND repay_status_apr > 0)
            THEN 1
            ELSE 0
        END AS continuous_overdue_flag_6m,

        COUNT(*) AS customer_count,
        AVG(default_flag) AS default_rate

    FROM credit_risk
    GROUP BY continuous_overdue_flag_6m;
    """,
    conn
)

,continuous_overdue_flag_6m,customer_count,default_rate
0,0,24773,0.154927
1,1,5227,0.535297


**分析结果：**
- 无连续逾期：24,773人，违约率约15.49%
- 存在连续逾期：5,227人，违约率约53.53%

连续逾期客户的下一期违约率明显更高，说明连续性能够在单月逾期之外进一步刻画历史还款风险。

## 7. SQL 分析总结

本 Notebook 使用 SQL 完成了从原始还款状态到风险指标的提取与验证，主要涉及：

- `SELECT / FROM / WHERE`
- `COUNT / SUM / AVG`
- `CASE WHEN`
- `GROUP BY`
- `AND / OR`
- SQLite 多参数 `MAX()`
- 多个 `CASE WHEN` 相加构造风险特征

### 核心结论
1. 最近一期逾期客户的下一期违约率明显高于未逾期客户；
2. 逾期程度和近6个月逾期频率越高，整体违约风险越高；
3. 最大逾期程度能够进一步反映历史风险严重性；
4. 连续逾期客户表现出明显更高的后续违约风险；
5. SQL 可以直接从底层账户字段中构造具有业务解释性的风险指标，并完成分组风险验证。

### 项目定位
该部分重点展示使用 SQL 进行消费信贷风险指标提取、特征构造和分组分析的能力。分析结论基于当前公开数据集，不代表生产环境中的正式风险策略。